In [1]:
def reasoning_steps_reward(completions, **kwargs):
    r"""
    奖励函数，用于鼓励清晰的逐步推理。
    它会查找诸如 "Step 1:"、编号列表、项目符号以及过渡词等模式。
    """
    # 定义正则表达式模式，用于查找推理步骤的指示符
    # 该模式可以匹配以下几种情况：
    # 1. "Step X:" 格式，其中 X 是数字
    # 2. 以数字加句号开头的行（编号列表）
    # 3. 以换行符加连字符开头的行（项目符号列表）
    # 4. 以换行符加星号开头的行（项目符号列表）
    # 5. 过渡词，如 "First,"、"Second,"、"Next,"、"Finally,"
    pattern = r"(Step \d+:|^\d+\.|\n-|\n\*|First,|Second,|Next,|Finally,)"

    # 从每个完成项中提取内容
    # completions 是一个列表，每个元素通常是一个列表，其中第一个元素是包含 "content" 键的字典
    # 这里通过列表推导式提取出每个完成项的内容
    completion_contents = [completion[0]["content"] for completion in completions]

    # 统计每个完成项中推理步骤指示符的数量
    # 使用 re.findall 函数结合 re.MULTILINE 标志，在多行模式下查找所有匹配的模式
    # 并计算匹配结果的数量
    matches = [len(re.findall(pattern, content, re.MULTILINE))
               for content in completion_contents]

    # 奖励值与推理步骤的数量成正比，最大奖励值为 1.0
    # 这里使用了一个“魔法数字” 3，意味着至少有 3 个推理步骤才能获得全额奖励
    # 通过 min 函数确保奖励值不会超过 1.0
    return [min(1.0, count / 3) for count in matches]

In [2]:
# 实现格式奖励函数
def format_reward(completions, **kwargs):
    """
    此奖励函数用于检查完成结果是否符合特定格式：<think>...</think> <answer>...</answer>。
    """
    # 定义所需格式的正则表达式模式
    # 该模式要求字符串以 <think> 开头，中间可以包含任意字符（非贪婪匹配），接着是 </think>
    # 之后可以有零个或多个空白字符，然后是 <answer>，再接着是任意字符（非贪婪匹配），最后以 </answer> 结尾
    pattern = r"^<think>.*?</think>\s*<answer>.*?</answer>$"

    # 从每个完成结果中提取内容
    # completions 是一个列表，每个元素通常是一个列表，其中第一个元素是包含 "content" 键的字典
    # 这里通过列表推导式提取出每个完成结果的内容
    completion_contents = [completion[0]["content"] for completion in completions]

    # 检查每个完成结果的内容是否与定义的模式匹配
    # 使用 re.match 函数，结合 re.DOTALL 和 re.MULTILINE 标志
    # re.DOTALL 使得 . 可以匹配包括换行符在内的任意字符
    # re.MULTILINE 影响 ^ 和 $ 的匹配行为，使其可以匹配每行的开头和结尾
    matches = [re.match(pattern, content, re.DOTALL | re.MULTILINE)
               for content in completion_contents]

    # 根据匹配结果给予奖励
    # 如果匹配成功，奖励为 1.0；否则奖励为 0.0
    return [1.0 if match else 0.0 for match in matches]
  

In [3]:
  
def accuracy_reward(completions, solution, **kwargs):
    """
    奖励函数，用于检查模型的响应在数学上是否等同于真实答案。
    使用 latex2sympy2 进行解析，使用 math_verify 进行验证。
    """
    # 从 completions 中提取模型的响应内容
    # completions 是一个列表，其中每个元素是一个包含字典的列表，字典中有 "content" 键存储响应内容
    contents = [completion[0]["content"] for completion in completions]
    # 用于存储每个响应对应的奖励值的列表
    rewards = []

    # 遍历模型的响应内容和对应的真实答案
    for content, sol in zip(contents, solution):
        # 解析真实答案
        # 使用 parse 函数，以 "first_match" 模式提取，使用 LatexExtractionConfig 配置
        gold_parsed = parse(sol, extraction_mode="first_match", 
                            extraction_config=[LatexExtractionConfig()])
        # 检查真实答案是否解析成功
        if gold_parsed:
            # 以宽松的归一化配置解析模型的答案
            answer_parsed = parse(
                content,
                extraction_config=[
                    LatexExtractionConfig(
                        # 配置归一化参数
                        normalization_config=NormalizationConfig(
                            # 不进行 nits 归一化
                            nits=False,
                            # 不处理格式错误的运算符
                            malformed_operators=False,
                            # 处理基本 LaTeX 表达式
                            basic_latex=True,
                            # 处理方程
                            equations=True,
                            # 处理所有带框的表达式
                            boxed="all",
                            # 处理单位
                            units=True,
                        ),
                        # 带框匹配的优先级
                        boxed_match_priority=0,
                        # 不尝试在没有锚点的情况下提取
                        try_extract_without_anchor=False,
                    )
                ],
                # 以 "first_match" 模式提取
                extraction_mode="first_match",
            )
            # 如果模型答案和真实答案匹配，奖励为 1.0，否则为 0.0
            reward = float(verify(answer_parsed, gold_parsed))
        else:
            # 如果真实答案无法解析，给予中性奖励 0.5
            reward = 0.5
            # 打印警告信息，提示真实答案解析失败
            print("Warning: Failed to parse gold solution:", sol)
        # 将当前响应的奖励值添加到奖励列表中
        rewards.append(reward)
    # 返回所有响应对应的奖励值列表
    return rewards